# 🧪 Laboratório do contrato de dados (`jobs/contract.py`)

**O que este notebook é:** uma bancada interativa para *entender e demonstrar* as funções do contrato — rodar cada etapa com dados fabricados, ver o resultado no meio do caminho e testar hipóteses suas.

**O que ele não é:** substituto da suíte pytest. O notebook *descobre*; o pytest *lembra* — todo caso interessante que você encontrar aqui deve virar um teste permanente em `tests/` (a rede de regressão que roda no CI, numa máquina limpa, sem depender da ordem das células).

**Pré-requisitos** (uma vez, ver `notebooks/README.md`): venv com `pyspark==3.5.1`, Java 17+ e, para a seção 5, `boto3`. Nada aqui toca S3/LocalStack — tudo roda em memória.

**Sumário**
1. Setup: SparkSession e imports
2. Helpers de fabricação de dados (os mesmos dos testes)
3. As etapas do contrato, uma a uma
4. O pipeline completo: do cru ao silver
5. Do gerador à quarentena: distribuição de rejeições
6. Exercícios

> Convenção da pasta: todo notebook roda **de cima a baixo** (`Run → Run All Cells`). Se se perder no estado, `Kernel → Restart Kernel and Run All`.

## 1. Setup

Mesma receita da fixture de `tests/test_contract.py`: sessão local com 2 cores e `shuffle.partitions=2` (o default de 200 partições só deixaria tudo lento com dados minúsculos). O `sys.path` aponta para `jobs/` porque o notebook vive em `notebooks/`.

In [ ]:
import sys

sys.path.insert(0, "../jobs")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[2]")
    .appName("lab-contrato")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
from contract import (
    RAW_SCHEMA,
    apply_contract,
    cast_types,
    deduplicate,
    shape_silver,
    split_valid_quarantine,
)

# ordem de aplicacao no job (bronze_to_silver.py):
# cast_types -> deduplicate -> apply_contract -> split_valid_quarantine -> shape_silver

## 2. Helpers de fabricação (os mesmos dos testes)

`_row()` é a **linha perfeita**: um evento válido em todos os campos. Cada experimento sabota **só** o campo que quer exercitar (`_row(amount_cents="abc")`) — assim fica óbvio qual regra causou o resultado.

`raw_df()` monta o DataFrame todo string, igual ao que chega da bronze (`RAW_SCHEMA`).

In [ ]:
COLS = ["event_id", "occurred_at", "customer_id", "merchant_id",
        "payment_method", "currency", "amount_cents", "status", "channel"]


def _row(**kwargs):
    base = {
        "event_id": "e1", "occurred_at": "2026-07-03T10:00:00", "customer_id": "cus_1",
        "merchant_id": "mer_1", "payment_method": "pix", "currency": "BRL",
        "amount_cents": "1000", "status": "approved", "channel": "app",
    }
    base.update(kwargs)
    return base


def raw_df(rows):
    return spark.createDataFrame(
        [tuple(str(r[c]) if r[c] is not None else None for c in COLS) for r in rows],
        schema=COLS,
    )


raw_df([_row()]).show(truncate=False)

## 3. As etapas do contrato, uma a uma

### 3.1 `cast_types` — cast tolerante

A promessa: valor malformado vira **NULL controlado**, não exceção às 3h da manhã. Sabotamos `amount_cents` e `occurred_at` e olhamos as colunas tipadas:

In [ ]:
df = cast_types(raw_df([
    _row(event_id="ok"),
    _row(event_id="quebrado", amount_cents="abc", occurred_at="31/02/2026 99:99"),
]))
df.select("event_id", "amount_cents", "amount_cents_typed",
          "occurred_at", "occurred_at_typed").show(truncate=False)

O `try_cast` devolveu `NULL` para `"abc"` e para a data impossível — e o job continua vivo. Esses NULLs serão capturados adiante pelas regras `amount_invalido` e `timestamp_invalido`.

Teste você: troque por `"1e3"`, `"  100 "`, `"-0"` e veja o que sobrevive ao cast.

### 3.2 `deduplicate` — determinística (e por que não `dropDuplicates`)

Duas versões do mesmo `event_id`: a das 08h (`declined`) e a das 09h (`approved`). A dedup do contrato **sempre** fica com a mais recente (janela por `event_id` ordenada por `occurred_at` desc):

In [ ]:
versoes = raw_df([
    _row(event_id="e1", occurred_at="2026-07-03T08:00:00", status="declined"),
    _row(event_id="e1", occurred_at="2026-07-03T09:00:00", status="approved"),
])

deduplicate(cast_types(versoes)).select("event_id", "occurred_at", "status").show()

In [ ]:
# o contraste: dropDuplicates mantem um registro ARBITRARIO -- o "primeiro
# encontrado", que depende de particionamento e paralelismo, nao do dado
versoes.dropDuplicates(["event_id"]).select("event_id", "occurred_at", "status").show()

Rode a célula do `dropDuplicates` algumas vezes (ou em outra máquina) e o resultado **pode** mudar — é esse não-determinismo que o contrato elimina. É também essa ordenação temporal que absorve o upsert do CDC (`ingest/generate_cdc_updates.py`) sem mudar nada no pipeline.

### 3.3 `apply_contract` — as seis regras, com acúmulo de motivos

Um evento que viola **três** regras ao mesmo tempo. A promessa: os três motivos no array, não só o primeiro — quem conserta a origem precisa da lista completa.

In [ ]:
tres_defeitos = raw_df([_row(amount_cents="-5", currency="xxx", status="unknown")])
apply_contract(cast_types(tres_defeitos)).select("event_id", "rejection_reasons").show(truncate=False)

### 3.4 `split_valid_quarantine` — nada é descartado

Array vazio = válido; array com motivo = quarentena. O reprovado **não some**: vira evidência para cobrar correção na origem.

In [ ]:
lote = apply_contract(cast_types(raw_df([
    _row(event_id="bom"),
    _row(event_id="sem_cliente", customer_id=None),
    _row(event_id="moeda_ruim", currency="BR"),
])))
validos, quarentena = split_valid_quarantine(lote)
print("validos:", validos.count(), "| quarentena:", quarentena.count())
quarentena.select("event_id", "rejection_reasons").show(truncate=False)

### 3.5 `shape_silver` — a modelagem final

Nomes de negócio, tipos fortes, colunas derivadas (`amount` em decimal, `occurred_hour`) e a linhagem técnica (`_ingested_at`, `_source_job`, `dt`):

In [ ]:
silver = shape_silver(validos, dt="2026-07-03", run_ts="2026-07-03T12:00:00")
silver.printSchema()
silver.show(truncate=False)

## 4. O pipeline completo: do cru ao silver

O mesmo encadeamento do job real (`bronze_to_silver.py`), numa célula — útil como demo de ponta a ponta:

In [ ]:
cru = raw_df([
    _row(event_id="a"),
    _row(event_id="a"),                                # duplicata exata
    _row(event_id="b", amount_cents=None),             # amount_invalido
    _row(event_id="c", occurred_at="ontem de manha"),  # timestamp_invalido
    _row(event_id="d", currency="EUR"),                # valido: EUR esta no dominio
])

checked = apply_contract(deduplicate(cast_types(cru)))
validos, quarentena = split_valid_quarantine(checked)

print(f"entrada: {cru.count()} | pos-dedup: {checked.count()} | "
      f"validos: {validos.count()} | quarentena: {quarentena.count()}")
quarentena.select("event_id", "rejection_reasons").show(truncate=False)

## 5. Do gerador à quarentena: a distribuição de rejeições

Agora em escala: os **mesmos** `clean_event`/`corrupt` do gerador (`ingest/generate_events.py`) fabricam alguns milhares de eventos em memória (sem S3), e medimos a distribuição de motivos na quarentena.

As probabilidades do `corrupt` são conhecidas — ~2% `amount`, ~1,2% moeda, ~1% timestamp, ~0,8% `customer_id` — então a distribuição observada deve ecoá-las. É a mesma conta que o quality gate faz com o `reject_rate`.

> Requer `boto3` no venv (o módulo do gerador importa boto3 no topo, mesmo sem usarmos S3 aqui).

In [ ]:
import random

sys.path.insert(0, "../ingest")
from generate_events import clean_event, corrupt

rng = random.Random(42)
merchant_ids = [f"mer_{i:05d}" for i in range(1, 301)]
eventos = [corrupt(clean_event(rng, "2026-07-03", merchant_ids), rng) for _ in range(4000)]

checked = apply_contract(deduplicate(cast_types(raw_df(eventos))))
validos, quarentena = split_valid_quarantine(checked)

total = checked.count()
rejeitados = quarentena.count()
print(f"total: {total} | rejeitados: {rejeitados} | reject_rate: {rejeitados / total:.2%}")

(quarentena
 .select(F.explode("rejection_reasons").alias("motivo"))
 .groupBy("motivo").count()
 .orderBy(F.desc("count"))
 .show())

## 6. Exercícios

1. **Regra nova na marra:** invente um evento com `channel` fora do domínio (`"telegrama"`). Ele passa? Por quê? Proponha a regra `channel_fora_dominio` (sem implementar) e discuta o custo de endurecer um contrato já em produção.
2. **Caso de borda do cast:** encontre um valor de `amount_cents` que *sobrevive* ao `try_cast` mas que o negócio consideraria inválido. Em qual regra ele deveria cair?
3. **Feche o ciclo:** escolha o caso mais interessante que você viu aqui e escreva-o como teste em `tests/test_contract.py`, no padrão `_row`/arrange-act-assert. Rode `make test` e confirme que a suíte inteira segue verde — é assim que a descoberta do notebook vira proteção permanente.

In [ ]:
# libere os recursos ao terminar (a JVM do Spark nao morre com o kernel ocioso)
spark.stop()